# Process Raw Data

This takes the raw data in the Model/rawdata folder and splits it along the detected 'line' between images

In [1]:
# Imports
import os
from pathlib import Path
from urllib.parse import urlparse

from fastai.vision.all import *
from fastai.vision.core import *

from PIL import ImageFilter, Image
import numpy as np

import ipyplot

from ipywidgets import IntProgress
from IPython.display import display

from ipynb.fs.full.Deink_00_Utils import *
from ipynb.fs.full.Deink_00_Utils import _get_sil_y 

Fastai Version: 2.7.12
Image Size: (480, 360)
Clean: True
Batch Size: 4


# Set some processing variables

In [2]:
# Should we clean out the destination directories before processing?
clean = True

# Set up util functions

In [3]:
# process a single image
def process_image(img):
    # Do the resizing
    resize_img = resize(img, split_idx=1) # Resize the image to match our training.
    
    return resize_img.convert('RGB')

In [4]:
# compile some regex for future matching

# These are for the data sets where the source originally had a tattoo and was then cleaned
## clean/tattoo naming pair convention
clean_re = re.compile(r'.*_clean\..*') # for images that had tattoos, but are cleaned now
tattoo_re = re.compile(r'.*_tattoo\..*') # for images with tattoos

# Process the images

In [5]:
# Should we clean first?
if clean:
    print(f"Cleaning {path_deinked_clean}")
    for x in path_deinked_clean.glob("retouchme-*"):
        if x.is_file():
            os.remove(x)
    for x in path_deinked_clean.glob("laser-removal-*"):
        if x.is_file():
            os.remove(x)
    for x in path_deinked_clean.glob("101000555image*"):
        if x.is_file():
            os.remove(x)
            
    print(f"Cleaning {path_deinked_tattoo}")
    for x in path_deinked_tattoo.glob("retouchme-*"):
        if x.is_file():
            os.remove(x)
    for x in path_deinked_tattoo.glob("laser-removal-*"):
        if x.is_file():
            os.remove(x)
    for x in path_deinked_tattoo.glob("101000555image*"):
        if x.is_file():
            os.remove(x)

Cleaning data/deinked/clean
Cleaning data/deinked/tattoo


In [6]:
# Fetch our files from the rawdata
raw_image_files = get_image_files('./data/deinked/rawdata')

max_count = len(raw_image_files)
f = IntProgress(min=0, max=max_count, description="Raw Image: ") # instantiate the bar
display(f) # display the bar

# loop over all the images and process them
for raw_image_file in raw_image_files:
    f.value += 1 # signal to increment the progress bar
    
    # load the image and process
    #print(f"Processing {raw_image_file}")
    pil_image = PILImage.create(raw_image_file)
    resize_img = process_image(pil_image)
    
    # Save the file
    basename = os.path.basename(raw_image_file)
    if clean_re.match(basename):
        resize_img.save(path_deinked_clean / re.sub(r'_clean', "", basename))
    elif tattoo_re.match(basename):
        resize_img.save(path_deinked_tattoo / re.sub(r'_tattoo', "", basename))
    else: 
        print(f"WARNING: {raw_image_file} does not match naming convention. Skipping.")

IntProgress(value=0, description='Raw Image: ', max=244)

/opt/conda/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:866: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 2. 
  warnings.warn(str(msg))
